# Asian forward on CMA curves

This notebook prices a strip of calendar-month-average (CMA) Asian forwards.

Python owns the averaging and settlement dates. The binding converts `datetime.date` values internally.

A `ForwardCurve` with `averaging=ForwardAveraging.CALENDAR_MONTH` treats observation dates by month and year only: every day in a month has the same forward. `VolatilityCurveCMA` is constructed to show that it is exposed, but `AsianForward` does not use volatility.

An Asian contract still requires a fixing source even when every averaging date is in the future.

Build the module before running the notebook (copies `_core.pyd` into `python/sjd`):

`cmake --build --preset debug --target sjd_python`

Install the package once with `python -m pip install -e python`.

Shut down the notebook kernel before rebuilding so Windows can overwrite `_core.pyd`.

In [1]:
import math
from datetime import date

import sjd
from sjd import examples

print(sjd.__file__)
print([name for name in dir(sjd) if not name.startswith("_")])


E:\Code\sjd\python\sjd\__init__.py
['AsianForward', 'AsianOption', 'Black76Call', 'Black76Put', 'BusinessDayConvention', 'BuySell', 'Calendar', 'Contract', 'CurveInterpolator', 'DailyFixingCurve', 'DiscountCurve', 'EuropeanForward', 'EuropeanOption', 'ForwardAveraging', 'ForwardCurve', 'ForwardRoll', 'Margin', 'Portfolio', 'PortfolioB2CEM', 'PortfolioEvolver', 'PutCall', 'RatesEnvironment', 'RegCapCreditParameters', 'RegCapUnderlyingFactors', 'RegulatoryCapital', 'SimpleMargin', 'Simulation', 'SingleFactorEconomicScenarioGenerator', 'SingleInstrumentContract', 'SurfaceInterpolator', 'TimeUnit', 'ValidationIssue', 'ValidationReport', 'ValidationSeverity', 'VolatilityCurve', 'VolatilityCurveCMA', 'VolatilityRateSource', 'VolatilitySurfaceDelta', 'VolatilitySurfaceMoneynessCMA', 'examples', 'reg_cap_factors', 'simulate']


## Market data

The first curve pillar is the valuation date; later pillars are month starts. The calendar-month forward curve then returns that month's rate for every averaging day in the month.

The Asian forward value, before any historic fixes, is:

`(average of remaining CMA forwards - strike) * domestic discount factor * volume * buy/sell sign`

In [2]:
anchor = examples.ANCHOR
curve_dates = [
    anchor,
    examples.month_start(2026, 10),
    examples.month_start(2026, 12),
    examples.month_start(2027, 3),
    examples.month_start(2027, 9),
]

forwards, discount_factors, volatilities = examples.fx_curve_values(curve_dates)

frs = sjd.ForwardCurve(
    anchor,
    curve_dates,
    forwards,
    allow_extrapolation=False,
    averaging=sjd.ForwardAveraging.CALENDAR_MONTH,
)
drs = sjd.DiscountCurve(
    anchor, curve_dates, discount_factors, allow_extrapolation=False
)
vrs = sjd.VolatilityCurveCMA(
    anchor, curve_dates, volatilities, allow_extrapolation=False
)

print("curves ok:", frs.is_ok(), drs.is_ok(), vrs.is_ok())
print("October CMA forward:", frs.get_forward(date(2026, 10, 16)))
print("same month, later day:", frs.get_forward(date(2026, 10, 30)))


curves ok: True True True
October CMA forward: 17.521588647801778
same month, later day: 17.521588647801778


In [3]:
# Historic fixes are required by AsianForward.validate even when unused.
fixing_dates = [date(2026, 8, 3), date(2026, 8, 31), date(2026, 9, 15)]
fixing_rates = [17.20, 17.35, 17.48]
fixings = sjd.DailyFixingCurve(fixing_dates, fixing_rates)

environment = sjd.RatesEnvironment(anchor)
environment.set_frs(frs)
environment.set_drs(drs)
environment.set_vrs(vrs)  # Not required by AsianForward.
environment.set_fixing_rs(fixings)

print("fixings ok:", fixings.is_ok())
print("environment:", environment.is_ok(), environment.error_messages())

fixings ok: True
environment: True 


## Define and validate the strip

Each trade averages the USDZAR business days in its contract month. Settlement is two business days after the last averaging date. A USDZAR business day is open in both the United States and South Africa (`sjd.Calendar`).

In [4]:
trade_inputs = [
    {
        "year": 2026,
        "month": 10,
        "strike": 17.40,
        "volume": 1_000_000.0,
        "side": sjd.BuySell.BUY,
    },
    {
        "year": 2026,
        "month": 12,
        "strike": 17.80,
        "volume": 750_000.0,
        "side": sjd.BuySell.SELL,
    },
    {
        "year": 2027,
        "month": 3,
        "strike": 18.10,
        "volume": 500_000.0,
        "side": sjd.BuySell.BUY,
    },
]

holidays = examples.sample_calendar()
strip = []
for trade in trade_inputs:
    averaging_dates = examples.business_days_in_month(holidays, trade["year"], trade["month"])
    settlement = holidays.advance(averaging_dates[-1], 2)
    forward = sjd.AsianForward(
        averaging_dates=averaging_dates,
        settlement=settlement,
        strike=trade["strike"],
        volume=trade["volume"],
        buy_sell=trade["side"],
    )
    forward.validate(environment)
    if not forward.is_ok():
        raise ValueError(forward.error_messages())
    strip.append((trade, averaging_dates, settlement, forward))

for trade, averaging_dates, settlement, forward in strip:
    print(
        f"{trade['year']}-{trade['month']:02d}  "
        f"{len(averaging_dates)} averaging days  "
        f"maturity {forward.maturity_date()}  "
        f"settlement {settlement}"
    )


2026-10  21 averaging days  maturity 2026-10-30  settlement 2026-11-03
2026-12  21 averaging days  maturity 2026-12-31  settlement 2027-01-05
2027-03  20 averaging days  maturity 2027-03-31  settlement 2027-04-02


In [5]:
print(f"{'Month':<8} {'Strike':>10} {'AveFwd':>10} {'DF':>10} {'Value':>15}")
rows = []
for trade, averaging_dates, settlement, forward in strip:
    average_forward = sum(frs.get_forward(d) for d in averaging_dates) / len(
        averaging_dates
    )
    discount_factor = drs.get_discount_factor(settlement)
    value = forward.get_value(environment)
    rows.append((trade, averaging_dates, settlement, forward, average_forward, discount_factor, value))
    print(
        f"{trade['year']}-{trade['month']:02d} {trade['strike']:10.4f} "
        f"{average_forward:10.4f} {discount_factor:10.6f} {value:15,.2f}"
    )

strip_value = sum(row[-1] for row in rows)
print(f"\nTotal strip value: {strip_value:,.2f}")

Month        Strike     AveFwd         DF           Value
2026-10    17.4000    17.5216   0.990854      120,476.55
2026-12    17.8000    17.6097   0.978972      139,755.19
2027-03    18.1000    17.7404   0.962825     -173,114.26

Total strip value: 87,117.49


In [6]:
# Check the binding against the C++ Asian-forward formula.
# With no averaging dates on or before the anchor, the historic-fixing
# contribution is zero and the average is the CMA forward for that month.
for trade, averaging_dates, settlement, forward, average_forward, discount_factor, value in rows:
    sign = 1.0 if trade["side"] == sjd.BuySell.BUY else -1.0
    expected = (average_forward - trade["strike"]) * discount_factor * trade["volume"] * sign
    assert math.isclose(value, expected, rel_tol=1e-12)
    first = frs.get_forward(averaging_dates[0])
    last = frs.get_forward(averaging_dates[-1])
    assert math.isclose(first, last, rel_tol=1e-12)

print("All strip values match the C++ Asian-forward formula.")
print("CMA forwards are constant inside each averaging month.")

All strip values match the C++ Asian-forward formula.
CMA forwards are constant inside each averaging month.
